# Tools

Models can reuqest to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are pairings of : 

1. A Schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute 

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model 
model = init_chat_model("groq:llama-3.3-70b-versatile")
response = model.invoke("What is life?")
response

AIMessage(content="What a profound and complex question. The concept of life is multifaceted and has been debated by philosophers, scientists, and scholars across various disciplines for centuries. While there is no single, definitive answer, I'll provide an overview of the different perspectives and aspects of life.\n\n**Biological definition:**\nFrom a biological perspective, life refers to the characteristic processes and functions that occur in living organisms, such as:\n\n1. Organization: Living organisms are composed of cells, which are the basic structural and functional units of life.\n2. Metabolism: Living organisms carry out chemical reactions to sustain themselves, grow, and reproduce.\n3. Homeostasis: Living organisms maintain a stable internal environment despite changes in external conditions.\n4. Growth and development: Living organisms increase in size, complexity, and functionality over time.\n5. Reproduction: Living organisms produce offspring, either sexually or ase

In [6]:
from langchain.tools import tool 

@tool
def get_weather(location:str)->str:
    """Get the weather at the locaation."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What's the weather in Hyderabad?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls by the model 
    print(f"Tools:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")
    

content='' additional_kwargs={'tool_calls': [{'id': 'd0r38ta3e', 'function': {'arguments': '{"location":"Hyderabad"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 223, 'total_tokens': 238, 'completion_time': 0.035072079, 'completion_tokens_details': None, 'prompt_time': 0.051244714, 'prompt_tokens_details': None, 'queue_time': 0.194569753, 'total_time': 0.086316793}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e457c-efdf-7f01-8653-d6977e3e3809-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Hyderabad'}, 'id': 'd0r38ta3e', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 223, 'output_tokens': 15, 'total_tokens': 238}
Tools:get_weather
Args:{'location': 'Hyderabad'}


# Tool Execution Loops

In [11]:
# Step 1 : Model generates tool calls 
messages = [{"role":"user","content":" What's the weather in Hyderabad?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 : Execute tools and collect results 
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generaed arguments 
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 : Pass results back to model for final response 
final_response = model_with_tools.invoke(messages)
print(final_response.text)


I'm glad I could help with the weather information. If you need more specific or up-to-date information, I can try to assist you further.


In [9]:
messages

[{'role': 'user', 'content': " What's the weather in Hyderabad?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'x3n9mj0hp', 'function': {'arguments': '{"location":"Hyderabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 223, 'total_tokens': 238, 'completion_time': 0.046010219, 'completion_tokens_details': None, 'prompt_time': 0.023373912, 'prompt_tokens_details': None, 'queue_time': 0.16012416, 'total_time': 0.069384131}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4593-720f-7c51-bdb4-f44305f601eb-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Hyderabad'}, 'id': 'x3n9mj0hp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 15, 'total_tokens': 238}),
 ToolMessage(cont